In [16]:
!git clone https://github.com/vietkhoanguyen050106-cmd/DistilBert_Restaurant.git
%cd DistilBert_Restaurant
!pip install -q transformers datasets accelerate scikit-learn

fatal: destination path 'DistilBert_Restaurant' already exists and is not an empty directory.
/kaggle/working/DistilBert_Restaurant/DistilBert_Restaurant


In [17]:
%cd /kaggle/working/DistilBert_Restaurant
!grep -n "_valid" src/data.py

/kaggle/working/DistilBert_Restaurant
8:def _valid(label):
14:    train_raw = ds["train"].filter(lambda e: _valid(e["label"]))
15:    test_raw = ds["test"].filter(lambda e: _valid(e["label"]))


In [18]:
%%writefile /kaggle/working/DistilBert_Restaurant/src/data.py
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

DATASET = "tomaarsen/setfit-absa-semeval-restaurants"


def _valid(label):
    return label is not None and str(label).strip() != ""


def build_dataloaders(tokenizer, max_len=128, batch_size=32, seed=42, val_size=0.1, test_size=0.1):
    ds = load_dataset(DATASET)
    train_raw = ds["train"].filter(lambda e: _valid(e["label"]))
    test_raw = ds["test"].filter(lambda e: _valid(e["label"]))

    label_names = sorted({str(l) for l in train_raw["label"]} | {str(l) for l in test_raw["label"]})
    l2i = {l: i for i, l in enumerate(label_names)}

    if len(test_raw) == 0:
        print("Test split không có nhãn -> tách test từ train.")
        s = train_raw.train_test_split(test_size=test_size, seed=seed)
        train_raw, test_raw = s["train"], s["test"]

    s = train_raw.train_test_split(test_size=val_size, seed=seed)
    train_raw, val_raw = s["train"], s["test"]

    def preprocess(ex):
        enc = tokenizer(ex["text"], ex["span"], truncation=True, max_length=max_len)
        enc["labels"] = l2i[str(ex["label"])]
        return enc

    sets = {n: d.map(preprocess, remove_columns=d.column_names)
            for n, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]}

    collator = DataCollatorWithPadding(tokenizer)
    make = lambda d, sh: DataLoader(d, batch_size=batch_size, shuffle=sh, collate_fn=collator)
    print(f"train={len(sets['train'])} val={len(sets['val'])} test={len(sets['test'])} labels={label_names}")
    return (make(sets["train"], True), make(sets["val"], False), make(sets["test"], False)), label_names

Overwriting /kaggle/working/DistilBert_Restaurant/src/data.py


In [19]:
!grep -n "_valid" /kaggle/working/DistilBert_Restaurant/src/data.py

8:def _valid(label):
14:    train_raw = ds["train"].filter(lambda e: _valid(e["label"]))
15:    test_raw = ds["test"].filter(lambda e: _valid(e["label"]))


In [20]:
%cd /kaggle/working/DistilBert_Restaurant
!rm -rf /kaggle/working/checkpoints
!python -m src.train_teacher --out /kaggle/working/checkpoints/teacher

/kaggle/working/DistilBert_Restaurant
Test split không có nhãn -> tách test từ train.
train=2990 val=333 test=370 labels=['conflict', 'negative', 'neutral', 'positive']
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2253.81it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED

In [22]:
!python -m src.train_student \
    --teacher_dir /kaggle/working/checkpoints/teacher \
    --out /kaggle/working/checkpoints/student_kd

Test split không có nhãn -> tách test từ train.
train=2990 val=333 test=370 labels=['conflict', 'negative', 'neutral', 'positive']
Student 67.0M params (random init)
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 2257.51it/s]
Layer mapping student->teacher: [(0, 0), (1, 2), (2, 4), (3, 6), (4, 8), (5, 10), (6, 12)]
[Student-KD] ep 01/40 loss=3.1889 ce=1.1116 kd=1.5233 hid=1.7891 | val_acc=0.5706 val_f1=0.1816
[Student-KD] ep 02/40 loss=2.5231 ce=1.0015 kd=1.2228 hid=1.3667 | val_acc=0.6366 val_f1=0.3235
[Student-KD] ep 03/40 loss=2.0885 ce=0.8240 kd=0.8541 hid=1.2434 | val_acc=0.6547 val_f1=0.3390
[Student-KD] ep 04/40 loss=1.8075 ce=0.7087 kd=0.6492 hid=1.1404 | val_acc=0.6607 val_f1=0.3744
[Student-KD] ep 05/40 loss=1.7399 ce=0.7060 kd=0.6343 hid=1.0841 | val_acc=0.6727 val_f1=0.4416
[Student-KD] ep 06/40 loss=1.5770 ce=0.6201 kd=0.5120 hid=1.0325 | val_acc=0.6937 val_f1=0.4688
[Student-KD] ep 07/40 loss=1.4333 ce=0.5414 kd=0.4101 hid=0.9838 | val_acc=0.6937 val_